In [ ]:
!pip install google-generativeai requests Pillow

In [ ]:
!pip install langchain-google-genai langchain_core pydantic


In [ ]:
#구글 드라이브 불러오기
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import google.generativeai as genai
import os

# API 키들을 로드하는 함수 (이미 작성하신 함수를 그대로 활용)
def load_api_keys(filepath="api_key.txt"):
    if not os.path.exists(filepath):
        print(f"오류: {filepath} 파일을 찾을 수 없습니다.")
        return

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

# 구글 드라이브 경로 설정
path = '/content/drive/MyDrive/capstone/'

# 1. api_key.txt로부터 모든 API 키 로드
load_api_keys(path + 'api_key.txt')

# 2. 환경변수에서 Gemini API 키 가져오기
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') # txt 파일 내 변수명에 맞게 수정하세요
if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
else:
    print("Gemini API Key가 파일에 없습니다.")

# 3. 환경변수에서 식품영양성분 API 키 가져오기
FOOD_NUTRITION_API_KEY = os.getenv('FOOD_NUTRITION_API_KEY')

if FOOD_NUTRITION_API_KEY:
    print("공공데이터 API 키가 성공적으로 로드되었습니다.")
else:
    print("FOOD_NUTRITION_API_KEY를 찾을 수 없습니다. api_key.txt를 확인해주세요.")

print("모든 API 설정이 완료되었습니다.")

In [ ]:
import google.generativeai as genai
import PIL.Image
import io
import json
import re
from urllib.parse import unquote

# 1. 초기 설정 및 모델 로드
# genai.configure(api_key="YOUR_API_KEY")
model = genai.GenerativeModel('gemini-2.5-flash')

# 공통적으로 사용할 프롬프트 (이미지용)
IMAGE_PROMPT = """이미지 속 식재료를 분석해서 JSON 배열로만 출력해.
1. "keyword": 수식어 없는 순수 이름 (예: 시금치, 배, 아보카도)
2. "group": (농축수산물, 음식, 가공식품) 중 선택
3. "subgroup": (과일류, 채소류, 곡류, 견과류, 해당없음) 중 가장 적절한 것 선택
반드시 ```json [ ... ] ``` 형식으로만 답해."""

# 텍스트 입력 시 구조화를 위한 프롬프트
TEXT_PROMPT = """다음 제공된 음식 리스트를 분석해서 JSON 배열로 변환해.
1. "keyword": 수식어 없는 순수 이름
2. "group": (농축수산물, 음식, 가공식품) 중 선택
3. "subgroup": (과일류, 채소류, 곡류, 견과류, 해당없음) 중 가장 적절한 것 선택
반드시 ```json [ ... ] ``` 형식으로만 답해.
입력값: {text_input}"""

# 2. 핵심 처리 함수 (이미지/텍스트 공통 처리)

def analyze_food(input_data, input_type="image"):
    """
    input_data: 이미지 바이트(bytes) 또는 음식명 문자열(str)
    input_type: "image" 또는 "text"
    """
    try:
        if input_type == "image":
            print(" Gemini가 이미지를 분석 중입니다...")
            img = PIL.Image.open(io.BytesIO(input_data))
            response = model.generate_content([IMAGE_PROMPT, img])

        elif input_type == "text":
            print(f" Gemini가 텍스트('{input_data}')를 분석 중입니다...")
            full_prompt = TEXT_PROMPT.format(text_input=input_data)
            response = model.generate_content(full_prompt)

        else:
            return {"error": "지원하지 않는 입력 타입입니다."}

        # JSON 파싱 공통 로직
        json_match = re.search(r'```json\n(.*?)\n```', response.text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(1))
        else:
            # 백업: 마크다운 기호가 없는 경우도 처리
            return json.loads(response.text)

    except Exception as e:
        return {"error": str(e)}

# 케이스 A: 이미지가 들어왔을 때 (현재 방식)
from google.colab import files
uploaded = files.upload()
if uploaded:
    img_bytes = list(uploaded.values())[0]
    result = analyze_food(img_bytes, input_type="image")
    print(f" 이미지 분석 결과: {result}")

# 케이스 B: 프론트에서 텍스트(음식명)가 들어왔을 때
user_text_input = "사과, 닭가슴살 샐러드, 아몬드"
result_text = analyze_food(user_text_input, input_type="text")
print(f" 텍스트 구조화 결과: {result_text}")

In [ ]:
import requests
import json
import re
import time
from urllib.parse import quote

raw_key = FOOD_NUTRITION_API_KEY.strip()
BASE_URL = "https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02"

final_results = []

print("\n" + "="*85)
print("🎯 [초정밀 매칭] 'in' 연산자 기반 중분류 교차 검증 가동")
print("="*85)

for item in parsed_food_list:
    keyword = item['keyword']
    target_sub = item.get('subgroup', '해당없음') # Gemini가 뽑은 '과일', '채소', '곡류' 등
    candidates = []

    encoded_q = quote(keyword)
    print(f"🔍 [{keyword} | {target_sub}] 탐색 중...", end=" ")

    # 5페이지(500개)를 뒤져서 가장 적합한 원재료를 찾습니다.
    for page in range(1, 14):
        url = f"{BASE_URL}?serviceKey={raw_key}&FOOD_NM_KR={encoded_q}&type=json&pageNo={page}&numOfRows=100"

        try:
            res = requests.get(url, timeout=10)
            if res.status_code != 200: break

            data = res.json()
            items_list = data.get('body', {}).get('items', [])
            if not isinstance(items_list, list):
                items_list = items_list.get('item', []) if isinstance(items_list, dict) else []

            if not items_list: break
            #점수를 도입 해서 공공데이터에서 좀더 정확하게 뽑아내기 위해 사용
            for it in items_list:
                fname = it.get('FOOD_NM_KR', '')
                db_sub = it.get('FOOD_CAT1_NM', '')

                score = 0

                # 1. [핵심] 중분류 'in' 매칭 (50만 점)
                if target_sub != "해당없음" and target_sub in db_sub:
                    score += 500000

                # 2. 이름 일치도 (배 vs 배, 신고, 생것 등 처리)
                if fname == keyword or fname.startswith(f"{keyword},") or fname.startswith(f"{keyword}_"):
                    score += 10000

                # 3. 이름 길이 패널티
                score -= (len(fname) - len(keyword)) * 1000

                #  [여기에 추가!] 원재료(생것/신선) 우선순위 점수 폭탄
                # 현재 음식 생것을 제대로 찾지 못하는 오류 발견
                if "_생것" in fname or ", 생것" in fname:
                    score += 1000000  # 100만 점을 더해 가공식품을 밀어냅니다.

                if score > 0:
                    candidates.append({'data': it, 'score': score})

            # 압도적인 후보(중분류 일치)를 찾았으면 다음 페이지는 보지 않음
            if any(c['score'] >= 500000 for c in candidates): break
        except:
            break

    if candidates:
        # 최종 점수가 가장 높은 단 하나를 선택
        best_match = max(candidates, key=lambda x: x['score'])['data']

        def to_f(v): return float(str(v).replace(",", "")) if v and v not in ["N/A", "string", ""] else 0.0

        # 필드명이 섞이지 않도록 통일된 딕셔너리 구조 사용
        res_dict = {
            'name': best_match.get('FOOD_NM_KR'),
            'cat': best_match.get('FOOD_CAT1_NM'),
            'cal': to_f(best_match.get('AMT_NUM1')),
            'carbo': to_f(best_match.get('AMT_NUM6')),
            'prot': to_f(best_match.get('AMT_NUM3')),
            'fat': to_f(best_match.get('AMT_NUM4'))
        }
        final_results.append(res_dict)
        print(f"성공 -> {res_dict['name']} ({res_dict['cat']})")
    else:
        print(f" 검색 실패 (분류 불일치)")

# 최종 리포트 출력
print("\n" + "="*85)
print(" [최종 결과] 정밀 필터링 완료 영양 리포트")
print("="*85)
for res in final_results:
    print(f"[{res['name']}] ({res['cat']}) | {res['cal']}kcal | 탄:{res['carbo']}g | 단:{res['prot']}g | 지:{res['fat']}g")

In [ ]:
import json
import re

# 3. [초정밀 교차 검증] API 결과와 Gemini 지식의 결합
# 공공데이터 에서 원하는 음식을 제대로 뽑아 내지못할때 최종 점검
# 추가적으로 100g 단위로 음식 성분 표시

final_nutrition_json = []

print("\n" + "="*85)
print("🤖 Gemini가 공공데이터를 검토하고 잘못된 정보를 수정 중입니다...")
print("="*85)

for i, item in enumerate(parsed_food_list):
    keyword = item['keyword']
    target_sub = item.get('subgroup', '해당없음')

    # 해당 키워드로 찾은 API 후보군을 가져옵니다 (현재 루프 내에서 처리 중인 데이터)
    # 실제로는 위 루프에서 candidates를 수집한 직후에 이 로직을 넣으시면 됩니다.

    # 제미나이에게 던질 최종 검증 프롬프트
    refine_prompt = f"""
    당신은 최고의 데이터 영양 분석가입니다.
    사용자가 찍은 사진 분석 결과와 공공데이터 API 검색 결과가 일치하지 않는 경우가 있습니다.

    [사용자 사진 분석 결과]: {keyword} ({target_sub})
    [공공데이터 검색 결과 후보들]: {json.dumps(candidates if 'candidates' in locals() else [], ensure_ascii=False)}

    [임무]
    1. 검색 결과 후보들 중 사진 속의 '{keyword}' 원재료(Fresh/Raw) 맥락과 가장 잘 맞는 항목을 선택하세요.
    2. 만약 후보들 중 '젤리', '과자', '음료' 등 가공식품만 있고 실제 과일/채소 데이터가 없다면,
       후보를 무시하고 당신이 알고 있는 '{keyword}'의 표준 100g당 영양 정보를 출력하세요.
    3. 모든 수치는 반드시 '100g' 기준으로 통일하세요.

    [출력 JSON 형식] (아래 형식만 엄격히 준수할 것):
    {{
      "food_name": "{keyword}",
      "is_corrected": "true/false (API 데이터가 틀려서 직접 수정했다면 true)",
      "cat": "식품분류",
      "cal": 0.0,
      "carbohydrate": 0.0,
      "protein": 0.0,
      "fat": 0.0
    }}
    """

    # 제미나이 호출
    correction_res = model.generate_content(refine_prompt)

    try:
        # 응답에서 JSON 구조만 추출
        json_str = re.search(r'\{.*\}', correction_res.text, re.DOTALL).group()
        refined_item = json.loads(json_str)
        final_nutrition_json.append(refined_item)

        status = "⚠️ AI 데이터 정정 완료" if refined_item['is_corrected'] == "true" else "✅ API 데이터 승인"
        print(f"🔍 [{keyword}] 처리 중... {status} -> {refined_item['cal']}kcal")
    except:
        print(f"❌ [{keyword}] 데이터 정제 중 오류 발생")

# 4. 최종 JSON 결과 출력
print("\n" + "="*85)
print(" [최종 결과] 정제된 탄단지 영양 JSON 데이터")
print("="*85)
final_output = json.dumps(final_nutrition_json, indent=2, ensure_ascii=False)
print(final_output)

In [ ]:
import os
import json
from pydantic import BaseModel, Field
from typing import List
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
#지금부터 음식 추천 agent
class FoodRecommendation(BaseModel):
    dish_name: str = Field(description="추천 요리 이름")
    additional_ingredients: List[str] = Field(description="필요한 추가 재료 및 간단한 활용 방법")
    health_benefits: str = Field(description="음식 효능 및 추천이유 간단 명료 하게 설명")
    recipe_tip: str = Field(description="간단한 조리 팁 또는 주의 사항 간략하게 설명")

class FoodRecommendationList(BaseModel):
    recommendations: List[FoodRecommendation]

In [ ]:
# 추천 agent 구축
class FoodRecommendationAgent:
    def __init__(self, api_key: str):
      self.llm = ChatGoogleGenerativeAI(
          model="gemini-2.5-flash",
          google_api_key=api_key,
          temperature=0.7
      )
      # json 출력 파서 초기화
      self.parser = JsonOutputParser(pydantic_object=FoodRecommendationList)

    def get_recommendation_chain(self):
      prompt = ChatPromptTemplate.from_template(
            "당신은 영양학과 요리에 정통한 AI 푸드 컨설턴트입니다.\n"
            "사용자가 현재 가지고 있는 식재료(또는 음식) 데이터를 분석하여, "
            "해당 재료들을 활용하거나 궁합이 좋은 요리 3가지를 추천해주세요.\n\n"
            "[현재 분석된 식재료 데이터]:\n{food_data}\n\n"
            "임무:\n"
            "1. 입력된 식재료를 최대한 활용할 수 있는 건강한 요리를 추천할 것.\n"
            "2. 각 요리별 영양학적 효능과 추천 이유를 명확히 설명할 것.\n"
            "3. 요리 초보자도 이해할 수 있는 간단한 조리 팁을 제공할 것.\n\n"
            "{format_instructions}"
      )
      # langchain 구성 : prompt -> llm -> output parser
      return prompt | self.llm | self.parser

In [ ]:
def generate_recommendations(analyzed_food_json: list):
  # 기존 로직에서 추출 분석 결과를 입력받아 추천결과 반환

  api_key = os.getenv("GEMINI_API_KEY")
  if not api_key:
      return {"error": "GEMINI_API_KEY가 설정되지 않았습니다."}

  print("👩‍🍳 AI 푸드 컨설턴트가 레시피를 고민 중입니다...")

  agent = FoodRecommendationAgent(api_key)
  chain = agent.get_recommendation_chain()

  try:
    result = chain.invoke({
        "food_data" : json.dumps(analyzed_food_json, ensure_ascii=False),
        "format_instructions": agent.parser.get_format_instructions()
    })
    return result
  except Exception as e:
    print(f"추천 중 오류 발생: {e}")
    return {"error" : str(e)}

In [ ]:
#현재 가상 데이터 를 넣고 음식을 추천받음
dummy_analyzed_data = [
    {"food_name": "시금치", "cat": "채소류", "cal": 23.0, "carbohydrate": 3.6, "protein": 2.9, "fat": 0.4},
    {"food_name": "계란", "cat": "농축수산물", "cal": 155.0, "carbohydrate": 1.1, "protein": 13.0, "fat": 11.0},
    {"food_name": "토마토", "cat": "채소류", "cal": 18.0, "carbohydrate": 3.9, "protein": 0.9, "fat": 0.2}
]

# 추천 함수 실행
recommendation_result = generate_recommendations(dummy_analyzed_data)

# 결과 출력
print("\n✨ [AI 추천 결과] ✨")
print(json.dumps(recommendation_result, indent=2, ensure_ascii=False))